In [49]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

# Generate synthetic data
np.random.seed(42)
n_samples = 10000
data = {
    'hour_cat': np.random.rand(n_samples) * 5,
    'hour_dog': np.random.rand(n_samples) * 5,
    'hour_rabbit': np.random.rand(n_samples) * 5,
    'post_category': np.random.choice(['cat', 'dog', 'rabbit'], n_samples),
}
df = pd.DataFrame(data)
# Probabilistic click generation with added randomness
def generate_click(row):
    if row['post_category'] == 'cat':
        return 1 if row['hour_cat'] + np.random.randn() > .5 else 0
    elif row['post_category'] == 'dog':
        return 1 if row['hour_dog'] + np.random.randn() > .5 else 0
    else:
        return 1 if row['hour_rabbit'] + np.random.randn() > .5 else 0

df['click'] = df.apply(generate_click, axis=1)

# Features and target
X = df.drop(columns=['click'])
y = df['click']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

In [50]:

# Custom PyTorch dataset class with preprocessing
# class ClickDataset(Dataset):
#     def __init__(self, X, y):
#         X = X.copy()
#         # Numerical preprocessing
#         for col in ['hour spent on reading cat post', 'hour spent on reading dog post', 'hour spent on reading rabbit post']:
#             X[col].fillna(X[col].mean(), inplace=True)
#             X[col] = (X[col] - X[col].mean()) / X[col].std()

#         # Categorical preprocessing
#         X['current post category'].fillna(X['current post category'].mode()[0], inplace=True)
#         X = pd.get_dummies(X, columns=['current post category'])

#         self.X = torch.tensor(X.values, dtype=torch.float32)
#         self.y = torch.tensor(y.values.reshape(-1, 1), dtype=torch.float32)

#     def __len__(self):
#         return len(self.y)

#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]
# # Create datasets
# train_dataset = ClickDataset(X_train, y_train)
# test_dataset = ClickDataset(X_test, y_test)

In [51]:
numericCols = [
    'hour_cat',
    'hour_dog',
    'hour_rabbit',
]
categoricalCols = ['post_category']

# Preprocessing functions
def preprocess_train(X):
    X = X.copy()
    numerical_means = {}
    numerical_stds = {}

    # Numerical preprocessing
    num_cols = ['hour_cat', 'hour_dog', 'hour_rabbit']
    for col in num_cols:
        numerical_means[col] = X[col].mean()
        numerical_stds[col] = X[col].std()
        X[col] = X[col].fillna(numerical_means[col])
        X[col] = (X[col] - numerical_means[col]) / numerical_stds[col]

    # Categorical preprocessing
    X['post_category'] = X['post_category'].astype('category')
    cat_categories = X['post_category'].cat.categories
    X = pd.get_dummies(X, columns=['post_category'])

    return X, numerical_means, numerical_stds, cat_categories

def preprocess_test(X, numerical_means, numerical_stds, cat_categories, train_cols):
    X = X.copy()
    num_cols = ['hour_cat', 'hour_dog', 'hour_rabbit']

    # Numerical preprocessing
    for col in num_cols:
        X[col] = X[col].fillna(numerical_means[col])
        X[col] = (X[col] - numerical_means[col]) / numerical_stds[col]

    # Categorical preprocessing using training categories
    X['post_category'] = pd.Categorical(X['post_category'], categories=cat_categories)
    X = pd.get_dummies(X, columns=['post_category'])

    # Handle new categories in test data
    for col in train_cols:
        if col not in X.columns:
            X[col] = 0
    X = X[train_cols]

    return X
# Preprocess datasets
X_train_processed, means, stds, cat_categories = preprocess_train(X_train)
X_test_processed = preprocess_test(X_test, means, stds, cat_categories, X_train_processed.columns)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_processed.values.astype(np.float32))
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1).astype(np.float32))
X_test_tensor = torch.tensor(X_test_processed.values.astype(np.float32))
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1).astype(np.float32))

# DataLoader
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=True)

In [52]:
# Define a simple DNN model
class DNN(nn.Module):
    def __init__(self, input_dim):
        super(DNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


# Model initialization
model = DNN(X_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 1
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

# Evaluation
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).numpy()
    y_pred_label = (y_pred > 0.5).astype(int)

In [53]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, fbeta_score

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_label)
precision = precision_score(y_test, y_pred_label)
recall = recall_score(y_test, y_pred_label)
f1 = f1_score(y_test, y_pred_label)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

f0_5 = fbeta_score(y_test, y_pred_label, beta=0.5)
f2 = fbeta_score(y_test, y_pred_label, beta=2.0)

print(f'F0.5 Score: {f0_5:.4f}')
print(f'F2 Score: {f2:.4f}')


Accuracy: 0.8640
Precision: 0.8640
Recall: 1.0000
F1 Score: 0.9270
F0.5 Score: 0.8882
F2 Score: 0.9695


In [54]:
classification_report(y_test, y_pred_label, output_dict=True)

/opt/homebrew/anaconda3/envs/python-notebook/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/homebrew/anaconda3/envs/python-notebook/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/homebrew/anaconda3/envs/python-notebook/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 272.0},
 '1': {'precision': 0.864,
  'recall': 1.0,
  'f1-score': 0.9270386266094421,
  'support': 1728.0},
 'accuracy': 0.864,
 'macro avg': {'precision': 0.432,
  'recall': 0.5,
  'f1-score': 0.46351931330472107,
  'support': 2000.0},
 'weighted avg': {'precision': 0.7464959999999999,
  'recall': 0.864,
  'f1-score': 0.8009613733905581,
  'support': 2000.0}}